## Crossover sensitivity

**Calculate percentage of patients that are high risk at different crossover thresholds: 14d, 30d, and 60dd**

In [1]:
import numpy as np
import pandas as pd

## Import data

In [2]:
treatment_df = pd.read_csv('../outputs/pembro_carbo_index.csv')

In [3]:
treatment_df.sample(3)

,PatientID,LineName,StartDate,avelumab_maintenance
792,FCC33E6A80709,pembro,2022-08-11,0
2556,F1D5EF5B471E5,carbo,2016-11-30,0
2315,F8C7A1F516ECE,carbo,2019-05-07,0


In [4]:
treatment_df.shape

(3712, 4)

In [5]:
treatment_df['treatment'] = (treatment_df['LineName'] == 'carbo').astype(int)

In [6]:
dtype_map = pd.read_csv('../outputs/pembro_carbo_features_dtypes.csv', index_col = 0).iloc[:, 0].to_dict()
features_df = pd.read_csv('../outputs/pembro_carbo_features_df.csv', dtype = dtype_map)

In [7]:
features_df.shape

(3706, 162)

In [8]:
surv_pred_df = pd.read_csv('../outputs/gb_6m_survival_predictions_calibrated.csv')

In [9]:
surv_pred_df.shape

(3468, 2)

In [10]:
df = pd.merge(features_df, treatment_df, on = 'PatientID', how = 'left')

In [11]:
df.shape

(3706, 166)

In [12]:
df = pd.merge(df, surv_pred_df, on = 'PatientID', how = 'left')

In [13]:
df.shape

(3706, 167)

In [14]:
df['StartDate'] = pd.to_datetime(df['StartDate'])

In [15]:
df['treatment_year'] = df['StartDate'].dt.year

In [16]:
df = df.query('treatment_year <= 2022')

In [17]:
df.shape

(3468, 168)

In [18]:
with open('../outputs/crossover_survival_estimate.txt', 'r') as f:
    crossover_survival_estimate_30 = float(f.read())

with open('../outputs/crossover_survival_estimate_14.txt', 'r') as f:
    crossover_survival_estimate_14 = float(f.read())

with open('../outputs/crossover_survival_estimate_60.txt', 'r') as f:
    crossover_survival_estimate_60 = float(f.read())

In [19]:
print(f'r* for 14d crossover: {crossover_survival_estimate_14}')
print(f'r* for 30d crossover: {crossover_survival_estimate_30}')
print(f'r* for 60d crossover: {crossover_survival_estimate_60}')

r* for 14d crossover: 0.6627284624635109
r* for 30d crossover: 0.4825724768396259
r* for 60d crossover: 0.1447800037948416


In [20]:
print(f'percent high risk at 14d crossover: {df.query('psurv_180_calibrated < @crossover_survival_estimate_14').shape[0]/df.shape[0]}')
print(f'percent high risk at 30d crossover: {df.query('psurv_180_calibrated < @crossover_survival_estimate_30').shape[0]/df.shape[0]}')
print(f'percent high risk at 60d crossover: {df.query('psurv_180_calibrated < @crossover_survival_estimate_60').shape[0]/df.shape[0]}')

percent high risk at 14d crossover: 0.37946943483275664
percent high risk at 30d crossover: 0.21539792387543252
percent high risk at 60d crossover: 0.0017301038062283738
